In [ ]:
pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()   # reads .env and makes the variables available
import os
key = os.environ["ELEVENLABS_API_KEY"]

In [ ]:
pip install elevenlabs

In [ ]:
import os, time, requests
from dotenv import load_dotenv

load_dotenv()   # loads .env from the notebook's working directory

API = "https://api.elevenlabs.io/v1/convai"
H = {"xi-api-key": os.environ["ELEVENLABS_API_KEY"]}

# 1) Upload the protocol as a knowledge base document
with open("protocol.pdf", "rb") as f:
    r = requests.post(
        f"{API}/knowledge-base/file",
        headers=H,
        files={"file": ("protocol.pdf", f, "application/pdf")},
        data={"name": "Study XYZ Protocol v1"},
    )
r.raise_for_status()
doc_id = r.json()["id"]
print("document id:", doc_id)

# 2) Trigger RAG indexing
r = requests.post(
    f"{API}/knowledge-base/{doc_id}/rag-index",
    headers={**H, "Content-Type": "application/json"},
    json={"model": "multilingual_e5_large_instruct"},
)
r.raise_for_status()
print("rag status:", r.json())

In [ ]:
# 1) Upload the protocol as a knowledge base document
with open("protocol.pdf", "rb") as f:
    r = requests.post(
        f"{API}/knowledge-base/file",
        headers=H,
        files={"file": ("protocol.pdf", f, "application/pdf")},
        data={"name": "Study XYZ Protocol v1"},
    )
print(r.status_code)
print(r.json())          # shows the full response so we can grab the right field
r.raise_for_status()
doc_id = r.json()["id"]
print("document id:", doc_id)

In [ ]:
# 2) Trigger RAG indexing (pick an allowed embedding model)
r = requests.post(
    f"{API}/knowledge-base/{doc_id}/rag-index",
    headers={**H, "Content-Type": "application/json"},
    json={"model": "multilingual_e5_large_instruct"},
)
r.raise_for_status()
print("rag status:", r.json())

# 3) Poll until indexing completes (large docs take a few minutes)
for _ in range(30):
    s = requests.get(f"{API}/knowledge-base/{doc_id}/rag-index", headers=H).json()
    print(s)
    if str(s).lower().find("succeeded") != -1 or str(s).lower().find("complete") != -1:
        break
    time.sleep(10)

In [ ]:
r = requests.post(
    f"{API}/knowledge-base/{doc_id}/rag-index",
    headers={**H, "Content-Type": "application/json"},
    json={"model": "e5_mistral_7b_instruct"},
)
print(r.status_code)
print(r.json())